In [9]:
import pathlib

import numpy as np
import pandas as pd

In [5]:
data_folder = pathlib.Path("../data/raw/")
metabolic_phenotypes_file = data_folder / "metabolic_phenotypes_bacdive.tsv"
non_metabolic_phenotypes_file = data_folder / "non_metabolic_phenotypes_bacdive.tsv"
rast_features_file = data_folder / "rast_features.tsv"
interpro_features_file = data_folder / "interpro_features.tsv"

## The Phenotype data

All the data was obtained from [BacDive](https://bacdive.dsmz.de/)

Phenotypes:
1. Metabolic phenotypes: 250
2. Non-metabolic phenotypes: 5

Genomes:
1. Metabolic phenotypes: 6808
2. Non-metabolic phenotypes: 4880

To download detailed genome information refer to: https://www.ncbi.nlm.nih.gov/datasets/docs/v2/how-tos/genomes/get-genome-metadata/

> NOTE:
> Phenotypes are either 0 (absent), 1 (present) or NA (not available)

In [19]:
met_df = pd.read_csv(metabolic_phenotypes_file, index_col=0, sep="\t")
met_phenotype_names = list(met_df.columns)[2:] # exclude species and domain
met_df

,species,domain,raffinose--builds_acid_from,melibiose--builds_acid_from,L-alanine--carbon_source,maltose--carbon_source,D-galactose--carbon_source,D-mannose--carbon_source,raffinose--carbon_source,cellobiose--carbon_source,...,3-hydroxybutyrate--carbon_source,turanose--carbon_source,ethanol--carbon_source,D-arabitol--carbon_source,L-arabitol--carbon_source,L-fucose--carbon_source,2-oxoglutarate--carbon_source,L-histidine--carbon_source,alpha-D-glucose--carbon_source,L-glutamate--carbon_source
Genomes,,,,,,,,,,,,,,,,,,,,,
GCF_000160075,Abiotrophia defectiva,Bacteria,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
GCF_003151135,Abyssibacter profundi,Bacteria,0.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
GCF_017377855,Acanthopleuribacter pedis,Bacteria,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
GCF_000376245,Acaricomes phytoseiuli,Bacteria,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
GCF_010131535,Acerihabitans arboris,Bacteria,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
GCF_003075035,Zobellella maritima,Bacteria,NaN,NaN,1.0,NaN,NaN,1.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0
GCF_003012795,Zobellella taiwanensis,Bacteria,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
GCF_009725985,Zobellia amurskyensis,Bacteria,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
non_met_df = pd.read_csv(non_metabolic_phenotypes_file, index_col=0, sep="\t")
non_met_df

,strain_id,gram_positive,motility,oxygen_tolerance_aerobe,spore_formation
GenomeID,,,,,
GCF_012427845,159798,0.0,NaN,NaN,NaN
GCF_014652935,133046,0.0,0.0,NaN,0.0
GCF_001688765,131145,NaN,NaN,0.0,NaN
GCF_000420025,17629,1.0,1.0,1.0,0.0
GCF_001884035,140962,NaN,NaN,NaN,1.0
...,...,...,...,...,...
GCF_000373965,13936,0.0,1.0,1.0,NaN
GCF_001439805,13084,NaN,NaN,1.0,NaN
GCF_003002065,8042,1.0,NaN,1.0,1.0


To find out how much the two datasets overlap we can do:

In [15]:
a = len(set(met_df.index) - set(non_met_df.index))
b = len(set(non_met_df.index) - set(met_df.index))
a, b

(2404, 491)

## Processing the feature datasets

1. [RAST](https://rast.nmpdr.org/): 6997 features for 8501 genomes
2. [Intepro](https://www.ebi.ac.uk/interpro/): 20124 features for 5385 genomes

In [6]:
rast_df = pd.read_csv(rast_features_file, index_col=0, sep="\t")
rast_df

,SSO:000020125__Murein DD-endopeptidase MepM,SSO:000033083__repeat protein,SSO:000005939__Phosphatidylglycerophosphatase A (EC 3.1.3.27),SSO:000004884__Methylmalonate-semialdehyde dehydrogenase (EC 1.2.1.27),"SSO:000020349__NADH:ubiquinone oxidoreductase, NADH-binding (51 kD) subunit",SSO:000043478__Magnesium citrate secondary transporter,"SSO:000000080__2,3-dihydro-2,3-dihydroxybenzoate dehydrogenase (EC 1.3.1.28)",SSO:000007884__TDP-N-acetylfucosamine:lipid II N-acetylfucosaminyltransferase (EC 2.4.1.325),SSO:000017709__Heme biosynthesis protein,"SSO:000041924__glycoside hydrolase, family 37",...,SSO:000007881__TATA-box binding protein,SSO:000021617__Phosphate-selective porin O and P,SSO:000009584__2OG-Fe(II) oxygenase,SSO:000012182__Chloride channel protein,SSO:000021803__Plasmid partitioning protein,SSO:000017551__HD-GYP domain protein,SSO:000002621__Exodeoxyribonuclease V beta chain (EC 3.1.11.5),SSO:000039031__Ribonucleotide monophosphatase NagD (EC 3.1.3.5),SSO:000032815__putative prophage repressor,"SSO:000001251__Bis(5'-nucleosyl)-tetraphosphatase, symmetrical (EC 3.6.1.41)"
GenomeID,,,,,,,,,,,,,,,,,,,,,
GCF_005938105,0,0,0,0,0,0,0,0,0,0,...,0,0,1,1,0,0,0,0,0,0
GCF_000284515,0,0,1,0,0,0,0,0,0,0,...,0,0,1,0,0,0,0,0,0,0
GCF_001708125,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
GCF_002224365,0,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
GCF_001746755,0,0,1,0,0,0,0,0,0,0,...,0,0,1,1,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
GCF_016863215,0,0,0,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
GCF_002899475,1,0,1,0,0,0,1,1,0,0,...,0,0,0,0,0,0,0,1,0,1
GCF_000242075,0,0,0,0,0,0,0,1,0,0,...,0,0,1,1,0,0,0,0,0,0


In [7]:
interpro_df = pd.read_csv(interpro_features_file, index_col=0, sep="\t")
interpro_df

,IPR016372__K88 fimbrial protein A,IPR023894__Sporulation delaying protein SdpB,"IPR022765__Dna2/Cas4, domain of unknown function DUF83","IPR024421__Bacteriophage P22, Mnt",IPR045517__Glyoxalase-related protein,IPR012489__Nuclease A inhibitor-like,"IPR024077__Neurolysin/Thimet oligopeptidase, domain 2","IPR026410__Oligosaccharyl transferase, archaeal",IPR018592__Protein of unknown function DUF2024,"IPR007646__RNA polymerase Rpb2, domain 4",...,IPR009513__PerB,IPR015942__Asp/Glu/hydantoin racemase,"IPR014330__RNA-binding S4-related,YaaA",IPR043931__Protein of unknown function DUF5779,IPR040572__Thaumarchaeal output domain 1,"IPR015170__Protein of unknown function DUF1924, Cytochrome c-type protein SHP-like","IPR014967__Uncharacterised protein family, YugN-like",IPR018758__Membrane iron-sulfur containing protein FtrD-like,IPR027826__Domain of unknown function DUF4431,"IPR015256__Initiation factor eIF2 gamma, C-terminal"
GenomeID,,,,,,,,,,,,,,,,,,,,,
GCF_004362145,0,0,0,0,0,0,1,0,0,0,...,0,1,0,0,0,0,0,0,0,0
GCF_000427095,0,0,0,0,0,0,0,0,0,0,...,0,1,1,0,0,0,0,0,0,0
GCF_009711225,0,0,0,0,1,0,1,0,0,0,...,0,1,0,0,0,0,0,0,0,0
GCF_900101385,0,0,0,0,0,0,0,0,0,0,...,0,1,1,0,0,0,0,0,0,0
GCF_900102145,0,0,1,0,0,0,0,0,0,0,...,0,1,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
GCF_900091575,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
GCF_003265355,0,0,0,0,0,0,1,0,0,0,...,0,1,0,0,0,0,0,0,0,0
GCF_000613725,0,0,0,0,0,0,1,0,0,0,...,0,1,0,0,0,0,0,0,0,0
